<a href="https://colab.research.google.com/github/kasrasa/Object-detection-tutorial/blob/YOLO/YOLO_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q -U pycocotools
!pip install -q -U ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 495.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 3.7 MB/s eta 0:00:00


In [64]:
import os
import random
import shutil
import urllib.request
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.ops import box_iou
from torchvision.transforms.functional import to_tensor
from ultralytics import YOLO

from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from pycocotools.coco import COCO
from IPython.display import display

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Torch version: 2.11.0+cpu
CUDA available: False


In [77]:
# -------------------------
# Global configuration
# -------------------------

SEED = 42
DEVICE = 0 if torch.cuda.is_available() else "cpu"  # Ultralytics accepts GPU index or "cpu"
TORCH_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# COCO annotation paths uploaded in Colab.
TRAIN_ANN = "/content/data/train/instances_train2014.json"
VAL_ANN = "/content/data/valid/instances_val2014.json"
TRAIN_INSTANCES_ROOT = Path("/content/data/train/")
VAL_INSTANCES_ROOT = Path("/content/data/valid/")
TRAIN_INSTANCES_ROOT.mkdir(parents=True, exist_ok=True)
VAL_INSTANCES_ROOT.mkdir(parents=True, exist_ok=True)

# Image roots. Images are downloaded on demand into these folders.
IMAGE_ROOT = Path("/content/data/images")
LABEL_ROOT = Path("/content/data/labels")
IMAGE_ROOT.mkdir(parents=True, exist_ok=True)
LABEL_ROOT.mkdir(parents=True, exist_ok=True)

# YOLO-native exported dataset root.
YOLO_DATA_ROOT = Path("/content/data/")
YOLO_ORIGINAL_ROOT = YOLO_DATA_ROOT / "original"
YOLO_EXPANDED_ROOT = YOLO_DATA_ROOT / "expanded"

# Dataset sizes.
NUM_TRAIN = 200
NUM_VAL = 50
MIN_SMALL_OBJECTS_PER_IMAGE = 3
NUM_ADDED_HARD_IMAGES = 100

# YOLO model and training settings.
# Change to "yolo11n.pt" or "yolov8n.pt" if you want an older baseline.
YOLO_WEIGHTS = "yolo26n.pt"
IMG_SIZE = 640
BATCH_SIZE = 8
NUM_WORKERS = 4
NUM_EPOCHS = 10
PATIENCE = 0
FREEZE_LAYERS = None  # Example: 10 to freeze early layers. None means no freeze argument.

# Evaluation / mining settings.
IOU_THRESH = 0.5 # regular iou threshold to match detected bbs to ground truth
SCORE_THRESH = 0.05 # intentionally low to see if model can detect objects with low confidence or completely misses them
POOR_RECALL_THRESHOLD = 0.7 # used to find weak classes and candidate classes for hard mining
MIN_SMALL_GT = 3 # number of small gt objects in the image that is not ocluded or crowded

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Using Ultralytics device:", DEVICE)
print("Train annotations exist:", os.path.exists(TRAIN_ANN))
print("Val annotations exist:", os.path.exists(VAL_ANN))

Using Ultralytics device: cpu
Train annotations exist: True
Val annotations exist: True


In [66]:
coco_train = COCO(TRAIN_ANN)
coco_val = COCO(VAL_ANN)

loading annotations into memory...
Done (t=12.69s)
creating index...
index created!
loading annotations into memory...
Done (t=5.09s)
creating index...
index created!


In [67]:
def build_coco_yolo_category_maps(coco):
  coco_to_yolo = {}
  yolo_to_coco = {}
  class_names = []
  cat_ids = sorted(coco.getCatIds())
  categories = coco.loadCats(cat_ids)
  for idx, cat in enumerate(categories):
    coco_to_yolo[cat["id"]] = idx # coco ids mapped to yolo
    yolo_to_coco[idx] = cat["id"] # yolo ids mapped to coco
    class_names.append(cat["name"])
  return coco_to_yolo, yolo_to_coco, class_names

def validate_ann(ann):
  if ann["iscrowd"] == 1:
    return False

  x, y, w, h = ann["bbox"]
  if ann.get("area", w*h) <= 1:
    return False
  if ann["bbox"][2] <= 1 or ann["bbox"][3] <= 1:
    return False
  return True

def coco_ann_to_yolo_row(ann, img_w, img_h, coco_to_yolo):
    x, y, w, h = ann["bbox"]

    cx = (x + w / 2) / img_w
    cy = (y + h / 2) / img_h
    bw = w / img_w
    bh = h / img_h

    class_id = coco_to_yolo[ann["category_id"]]

    return [class_id, cx, cy, bw, bh]

def create_yolo_dataset(anns, coco, coco_to_yolo):
  yolo_dataset = defaultdict(list)

  for ann in anns:
    image_id = ann["image_id"]
    image_info = coco.loadImgs(image_id)[0]

    img_w, img_h = image_info["width"], image_info["height"]
    yolo_row = coco_ann_to_yolo_row(ann, img_w, img_h, coco_to_yolo)
    yolo_dataset[image_id].append(yolo_row)
  return yolo_dataset

coco_to_yolo, yolo_to_coco, class_names = build_coco_yolo_category_maps(coco_train)
anns = coco_train.loadAnns(coco_train.getAnnIds())
yolo_dataset_train = create_yolo_dataset(anns, coco_train, coco_to_yolo)
coco_to_yolo, yolo_to_coco, class_names = build_coco_yolo_category_maps(coco_val)
anns = coco_val.loadAnns(coco_val.getAnnIds())
yolo_dataset_val = create_yolo_dataset(anns, coco_val, coco_to_yolo)
print("train images:", len(yolo_dataset_train))
print("val images:", len(yolo_dataset_val))

train images: 82081
val images: 40137


In [68]:
def classify_bb_area(anns):
  buckets = {
      "small": defaultdict(list),
      "medium": defaultdict(list),
      "large": defaultdict(list),
      "all": defaultdict(list)
  }

  for ann in anns:
    if not validate_ann(ann):
      continue

    image_id = ann["image_id"]
    area = ann["area"]

    if area < 32*32:
      buckets["small"][image_id].append(ann)
    elif area < 96*96:
      buckets["medium"][image_id].append(ann)
    else:
      buckets["large"][image_id].append(ann)

    buckets["all"][image_id].append(ann)
  return buckets

def get_images_by_object_size(
    coco,
    size_bucket,
    category_ids=None,
    min_objects=MIN_SMALL_OBJECTS_PER_IMAGE,
):
    selected_image_ids = []

    image_ids = coco.getImgIds()
    ann_ids = coco.getAnnIds(imgIds=image_ids, iscrowd=False)
    anns = coco.loadAnns(ann_ids)

    buckets = classify_bb_area(anns)

    for image_id, bucket_anns in buckets[size_bucket].items():
      if len(bucket_anns) >= min_objects:
        for ann in bucket_anns:
          if category_ids is not None and ann["category_id"] not in category_ids:
            continue

          selected_image_ids.append(image_id)

    return selected_image_ids

def train_val_images(selected_image_ids_train, selected_image_ids_val, num_train, num_val, seed = SEED):
  random.seed(seed)

  train_image_ids = []
  val_image_ids = []

  train_image_ids = random.sample(selected_image_ids_train, min(num_train, len(selected_image_ids_train)))
  val_image_ids = random.sample(selected_image_ids_val, min(num_val, len(selected_image_ids_val)))

  return train_image_ids, val_image_ids

selected_image_ids = get_images_by_object_size(coco_train, "all")
print("selected images:", len(selected_image_ids))
print(selected_image_ids[:10])
yolo_dataset_train[selected_image_ids[0]]

selected_image_ids_train = get_images_by_object_size(coco_train, "all", min_objects=MIN_SMALL_OBJECTS_PER_IMAGE)
selected_image_ids_val = get_images_by_object_size(coco_val, "all", min_objects=MIN_SMALL_OBJECTS_PER_IMAGE)
selected_image_ids_train, selected_image_ids_val = train_val_images(
    selected_image_ids_train, selected_image_ids_val,
    NUM_TRAIN, NUM_VAL
)
print("train images:", len(selected_image_ids_train))
print("val images:", len(selected_image_ids_val))

selected images: 558503
[57870, 57870, 57870, 57870, 57870, 57870, 57870, 57870, 57870, 57870]
train images: 200
val images: 50


In [81]:
def create_yolo_txt_files(label_path, coco, image_ids, yolo_dataset, split = "train"):
  dataset_root = Path(label_path)

  label_dir = dataset_root / split

  label_dir.mkdir(parents=True, exist_ok=True)

  for image_id in image_ids:
    image_info = coco.loadImgs(image_id)[0]
    file_name = image_info["file_name"]

    label_file = label_dir / file_name.replace(".jpg", ".txt")

    yolo_rows = yolo_dataset.get(image_id,[])
    if len(yolo_rows) == 0:
      print(f"Warning: no labels for image_id={image_id}, file={file_name}")

    with open(label_file, "w", encoding="utf-8") as f:
      for row in yolo_rows:
        class_id, cx, cy, bw, bh = row
        f.write(
                    f"{int(class_id)} "
                    f"{cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}\n"
                )

def download_image_file(image_path, coco, image_ids, split = "train"):
  dataset_root = Path(image_path)

  image_dir = dataset_root / split

  image_dir.mkdir(parents=True, exist_ok=True)

  for image_id in image_ids:
    image_info = coco.loadImgs(image_id)[0]
    file_name = image_info["file_name"]

    image_file = image_dir / file_name

    if not image_file.exists():
      image_url = f"http://images.cocodataset.org/{split}2014/{file_name}"
      urllib.request.urlretrieve(image_url, image_file)


def write_yolo_yaml(dataset_path, class_names):
  root = Path(dataset_path)

  with open(root/"data.yaml", "w", encoding="utf-8") as f:
    f.write(
        f"path: {dataset_path}\n"
        f"train: images/train\n"
        f"val: images/val\n"
        f"nc: {len(class_names)}\n"
        f"names: {class_names}\n"
    )

# create_yolo_txt_files(LABEL_ROOT, coco_train, selected_image_ids_train, yolo_dataset_train)
# create_yolo_txt_files(LABEL_ROOT, coco_val, selected_image_ids_val, yolo_dataset_val, "val")
# download_image_file(IMAGE_ROOT, coco_train, selected_image_ids_train)
# download_image_file(IMAGE_ROOT, coco_val, selected_image_ids_val, "val")
_, _, class_names = build_coco_yolo_category_maps(coco_train)
write_yolo_yaml(YOLO_DATA_ROOT, class_names)
label_files = list((Path(LABEL_ROOT) / "train").glob("*.txt"))
print("label files:", len(label_files))

first_label = label_files[0]
print("first label file:", first_label)

with open(first_label, "r") as f:
    print(f.read())

label files: 199
first label file: /content/data/labels/train/COCO_train2014_000000487602.txt
56 0.112667 0.659430 0.225333 0.285234
0 0.657677 0.503375 0.596271 0.777531
33 0.791177 0.547078 0.271271 0.688625
33 0.326250 0.678422 0.314167 0.565531
73 0.081115 0.212000 0.021813 0.039563
73 0.102646 0.047953 0.008625 0.035156
73 0.109469 0.050359 0.008687 0.033344
73 0.181760 0.167133 0.015396 0.042047
73 0.147104 0.107031 0.290333 0.063719
73 0.207656 0.220805 0.033438 0.031328
73 0.146115 0.222031 0.022937 0.029937
73 0.193927 0.064016 0.009646 0.032813
73 0.155510 0.217625 0.025687 0.044469
73 0.122438 0.266352 0.093875 0.053016
73 0.237896 0.166734 0.014583 0.047000
56 0.939719 0.858047 0.120562 0.279250
73 0.155479 0.164797 0.018083 0.042156
0 0.369969 0.629055 0.272146 0.698891
73 0.148958 0.130469 0.293750 0.242188

